# DDA-ViT Combined Comparison

## Modules

In [1]:
import os
import sys
import time
import random
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
import timm
from pathlib import Path
from torchvision import datasets, transforms

e:\Studies\MIT\8\Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configs

### Directories

In [2]:
ROOT_DIR = Path(os.getcwd()).resolve().parent.parent
STEEL_MODEL_PATH = ROOT_DIR / "Models" / "Final" / "Steel" / "steel.pth"
SUGAR_MODEL_PATH = ROOT_DIR / "Models" / "Final" / "Sugar" / "sugar.pth"
STEEL_TEST_DIR = ROOT_DIR / "Datasets" / "steel-defect-detection" / "test_images"
SUGAR_TEST_DIR = ROOT_DIR / "Datasets" / "sugar-quality-inspection" / "test_images"

print(f"Root:  {ROOT_DIR}")
print(f"Steel model:  {STEEL_MODEL_PATH}  exists={STEEL_MODEL_PATH.exists()}")
print(f"Sugar model:  {SUGAR_MODEL_PATH}  exists={SUGAR_MODEL_PATH.exists()}")
print(f"Steel test:   {STEEL_TEST_DIR}    exists={STEEL_TEST_DIR.exists()}")
print(f"Sugar test:   {SUGAR_TEST_DIR}    exists={SUGAR_TEST_DIR.exists()}")

Root:  E:\Studies\MIT\8\Project
Steel model:  E:\Studies\MIT\8\Project\Models\Final\Steel\steel.pth  exists=True
Sugar model:  E:\Studies\MIT\8\Project\Models\Final\Sugar\sugar.pth  exists=True
Steel test:   E:\Studies\MIT\8\Project\Datasets\steel-defect-detection\test_images    exists=True
Sugar test:   E:\Studies\MIT\8\Project\Datasets\sugar-quality-inspection\test_images    exists=True


### Constants

In [3]:
STEEL_CLASSES = ["1", "2", "3", "4"]
SUGAR_CLASSES = ["unsaturated", "metastable", "intermediate", "labile"]
NUM_STEEL_CLASSES = 4
NUM_SUGAR_CLASSES = 4
STEEL_IMAGE_SIZE = 256
SUGAR_IMAGE_SIZE = 224
ITERS = 5
STEEL_SAMPLES = 50

### Device

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

Device: cuda



## DDA-ViT

### Helpers

In [5]:
class SteelBackbone(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder = model.encoder
    def forward(self, x):
        features = self.encoder(x)
        return features[-1]

class SugarBackbone(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model.forward_features(x)

class CrossDomainAttention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
    def forward(self, Fs, Fq):
        out, _ = self.attn(Fs, Fq, Fq)
        return out

### Architecture

In [6]:
class DDAViT(nn.Module):
    def __init__(self, steel_model, sugar_model, embed_dim=256,
                 num_defect_classes=4, num_quality_classes=4):
        super().__init__()
        self.steel = SteelBackbone(steel_model)
        self.sugar = SugarBackbone(sugar_model)
        self.proj_s = nn.Conv2d(512, embed_dim, kernel_size=1)
        self.proj_q = nn.Linear(768, embed_dim)
        self.cross_attn = CrossDomainAttention(embed_dim)
        self.seg_head = nn.Conv2d(embed_dim, num_defect_classes, kernel_size=1)
        self.sugar_head = nn.Linear(embed_dim, num_quality_classes)

    def forward(self, x_steel=None, x_sugar=None):
        Fs_map, Fq = None, None
        if x_steel is not None:
            Fs_map = self.steel(x_steel)
            Fs_map = self.proj_s(Fs_map)
            B, d, H, W = Fs_map.shape
            Fs = Fs_map.flatten(2).transpose(1, 2)
        if x_sugar is not None:
            Fq = self.sugar(x_sugar)
            if Fq.dim() == 4:
                B, H, W, C = Fq.shape
                Fq = Fq.view(B, H * W, C)
            if Fq.dim() == 3:
                Fq = self.proj_q(Fq)
                Fq = Fq.mean(dim=1, keepdim=True)
            elif Fq.dim() == 2:
                Fq = self.proj_q(Fq)
                Fq = Fq.unsqueeze(1)
            else:
                raise ValueError(f"Unexpected sugar feature shape: {Fq.shape}")
        if Fs_map is not None and Fq is not None:
            Fs_fused = self.cross_attn(Fs, Fq)
            Fs_map = Fs_fused.transpose(1, 2).reshape(B, d, H, W)
        if x_steel is not None:
            seg_out = self.seg_head(Fs_map)
            seg_out = F.interpolate(
                seg_out,
                size=x_steel.shape[2:],
                mode="bilinear",
                align_corners=False
            )
            return seg_out
        if x_sugar is not None:
            F_out = Fq.mean(dim=1)
            return self.sugar_head(F_out)
        raise ValueError("Provide x_steel or x_sugar")

## Prediction Helpers

### Load DDA-ViT

In [7]:
def load_dda_vit():
    print("Loading DDA-ViT model...")
    steel_state = torch.load(str(STEEL_MODEL_PATH), map_location=device)
    sugar_state = torch.load(str(SUGAR_MODEL_PATH), map_location=device)
    steel_model = smp.Unet(
        encoder_name="mit_b4",
        encoder_weights=None,
        in_channels=3,
        classes=NUM_STEEL_CLASSES
    )
    steel_model.load_state_dict(steel_state, strict=False)
    sugar_model = timm.create_model(
        "swin_tiny_patch4_window7_224",
        pretrained=False,
        num_classes=NUM_SUGAR_CLASSES
    )
    sugar_model.load_state_dict(sugar_state, strict=False)
    for p in steel_model.parameters():
        p.requires_grad = False
    for p in sugar_model.parameters():
        p.requires_grad = False
    model = DDAViT(steel_model, sugar_model)
    model.to(device)
    model.eval()
    print("DDA-ViT loaded successfully.\n")
    return model

### Image Utilities

In [8]:
def load_sugar_image(path, size=SUGAR_IMAGE_SIZE):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Failed to load: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_h, new_w = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (new_w, new_h))
    pad_img = np.zeros((size, size, 3), dtype=np.uint8)
    pad_img[:new_h, :new_w] = resized
    img_norm = pad_img.astype(np.float32) / 255.0
    tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0)
    return tensor

In [9]:
def load_steel_image(path, patch_size=STEEL_IMAGE_SIZE):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Failed to load: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = cv2.resize(img, (img.shape[1], patch_size))
    return img

In [10]:
def sliding_window_inference(model, image, patch_size=256, stride=256):
    model.eval()
    H, W, _ = image.shape
    full_mask = np.zeros((4, H, W))
    count_map = np.zeros((H, W))
    for x in range(0, W - patch_size + 1, stride):
        patch = image[:, x:x + patch_size]
        patch_tensor = torch.from_numpy(patch).permute(2, 0, 1).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(x_steel=patch_tensor)
            probs = torch.sigmoid(output)[0].cpu().numpy()
        full_mask[:, :, x:x + patch_size] += probs
        count_map[:, x:x + patch_size] += 1
    full_mask /= np.maximum(count_map, 1e-6)
    return full_mask

### Sugar Prediction Function

In [11]:
def predict_sugar_batch(model, test_dir):
    all_preds = []
    all_labels = []
    class_names = sorted(os.listdir(str(test_dir)))
    for cls_idx, cls_name in enumerate(class_names):
        cls_dir = test_dir / cls_name
        if not cls_dir.is_dir():
            continue
        image_files = sorted([
            f for f in cls_dir.iterdir()
            if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
        ])
        for img_path in image_files:
            try:
                tensor = load_sugar_image(img_path)
                tensor = tensor.to(device)
                with torch.no_grad():
                    output = model(x_sugar=tensor)
                    probs = torch.softmax(output, dim=1)[0].cpu().numpy()
                pred = int(np.argmax(probs))
                all_preds.append(pred)
                all_labels.append(cls_idx)
            except Exception as e:
                print(f"  [WARN] Skipped {img_path.name}: {e}")
    return np.array(all_preds), np.array(all_labels), class_names

### Steel Prediction Function

In [12]:
def predict_steel_batch(model, test_dir, sample_limit=50):
    image_files = sorted([
        f for f in Path(test_dir).iterdir()
        if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    ])
    if sample_limit and len(image_files) > sample_limit:
        image_files = random.sample(image_files, sample_limit)
    defect_counts = {c: 0 for c in STEEL_CLASSES}
    total_images = 0
    for img_path in image_files:
        try:
            img = load_steel_image(img_path)
            probs = sliding_window_inference(model, img)
            mask = np.argmax(probs, axis=0)
            unique = np.unique(mask)
            for u in unique:
                if u > 0 and u <= 4:
                    defect_counts[str(int(u))] += 1
            total_images += 1
        except Exception as e:
            print(f"  [WARN] Skipped {img_path.name}: {e}")
    return total_images, defect_counts

## Prediction

In [13]:
model = load_dda_vit()
print("=" * 80)
print("DDA-ViT COMBINED COMPARISON")
print(f"Iterations: {ITERS}  |  Steel samples/iter: {STEEL_SAMPLES}")
print("=" * 80)

sugar_accuracies = []
per_class_accuracies = {cls: [] for cls in SUGAR_CLASSES}
steel_summaries = []
for i in range(1, ITERS + 1):
    print(f"\n{'─' * 60}")
    print(f"  ITERATION {i}/{ITERS}")
    print(f"{'-' * 60}")
    iter_start = time.time()
    print("  [Steel] Running predictions...")
    steel_total, steel_defects = predict_steel_batch(
        model, STEEL_TEST_DIR, sample_limit=STEEL_SAMPLES
    )
    steel_summaries.append({
        "iteration": i,
        "total_images": steel_total,
        "defect_counts": steel_defects
    })
    print(f"  [Steel] {steel_total} images | defects: {steel_defects}")
    print("  [Sugar] Running predictions...")
    preds, labels, class_names = predict_sugar_batch(model, SUGAR_TEST_DIR)
    accuracy = np.mean(preds == labels) * 100
    sugar_accuracies.append(accuracy)
    for cls_idx, cls_name in enumerate(class_names):
        cls_mask = (labels == cls_idx)
        if cls_mask.sum() > 0:
            cls_acc = np.mean(preds[cls_mask] == labels[cls_mask]) * 100
            per_class_accuracies[cls_name].append(cls_acc)
        else:
            per_class_accuracies[cls_name].append(0.0)
    iter_time = time.time() - iter_start
    print(f"  [Sugar] Accuracy: {accuracy:.2f}%")
    for cls_name in class_names:
        cls_acc = per_class_accuracies[cls_name][-1]
        print(f"          {cls_name:>14s}: {cls_acc:.2f}%")
    print(f"  Time: {iter_time:.1f}s")

print("\n" + "=" * 80)
print("SUMMARY — DDA-ViT (COMBINED)")
print("=" * 80)
print(f"\nSugar Accuracy over {ITERS} iterations:")
for i, acc in enumerate(sugar_accuracies, 1):
    print(f"  Iter {i:>2d}: {acc:.2f}%")
print(f"\n  Mean:  {np.mean(sugar_accuracies):.2f}%")
print(f"  Std:   {np.std(sugar_accuracies):.2f}%")
print(f"  Min:   {np.min(sugar_accuracies):.2f}%")
print(f"  Max:   {np.max(sugar_accuracies):.2f}%")
print(f"\nPer-Class Mean Accuracy:")
for cls_name in SUGAR_CLASSES:
    accs = per_class_accuracies[cls_name]
    if accs:
        print(f"  {cls_name:>14s}: {np.mean(accs):.2f}% (±{np.std(accs):.2f}%)")
print(f"\nSteel Defect Summary (sampled {STEEL_SAMPLES} images/iter):")
for summary in steel_summaries:
    print(f"  Iter {summary['iteration']:>2d}: {summary['defect_counts']}")

Loading DDA-ViT model...
DDA-ViT loaded successfully.

DDA-ViT COMBINED COMPARISON
Iterations: 5  |  Steel samples/iter: 50

────────────────────────────────────────────────────────────
  ITERATION 1/5
------------------------------------------------------------
  [Steel] Running predictions...
  [Steel] 50 images | defects: {'1': 50, '2': 1, '3': 1, '4': 0}
  [Sugar] Running predictions...
  [Sugar] Accuracy: 23.50%
            intermediate: 0.00%
                  labile: 42.00%
              metastable: 52.00%
             unsaturated: 0.00%
  Time: 24.5s

────────────────────────────────────────────────────────────
  ITERATION 2/5
------------------------------------------------------------
  [Steel] Running predictions...
  [Steel] 50 images | defects: {'1': 50, '2': 0, '3': 0, '4': 0}
  [Sugar] Running predictions...
  [Sugar] Accuracy: 23.50%
            intermediate: 0.00%
                  labile: 42.00%
              metastable: 52.00%
             unsaturated: 0.00%
  Time: 